# AI agents: pi on firehorse

In lesson 1 you used a **chat assistant**: you ask, it answers with text,
and *you* run the code. Today you meet an **agent**, which runs the code
itself.

Work through this notebook with a terminal open next to it. The commands
from "Install pi" on are typed on firehorse. Log in first (week 6):

```
server$ ssh yourname@firehorse.strawlab.org
```

## Chat assistant or agent?

| | Chat assistant | Agent |
| --- | --- | --- |
| Example | openwebui in the browser | pi, Claude Code |
| What it does | text in, text out | reads and writes files, runs commands, reads the errors, tries again |
| Who runs the code | **you**, by copy and paste | the agent itself |
| Sees your files | only what you paste | yes, directly |

An agent is a program that runs on a computer, here on firehorse. It sends
your request to a language model. The model answers with requests to use
**tools**, and the agent carries them out. pi has four tools:

- `read`: read a file,
- `write`: write a new file,
- `edit`: change a file,
- `bash`: run a command in the terminal, like the ones you typed in weeks
  5 and 6.

The agent shows the result to the model, and the loop goes on until the
task is done. It is the loop from lesson 1 (ask, read, run, test), but the
agent runs it by itself.

## Other agents

We use **pi** (https://pi.dev) because it is small, free, and works with
the university models. There are many others, for example **opencode**,
**Hermes**, **Claude Code** (Anthropic) and **OpenAI Codex**. They all work
in the same way, so what you learn here carries over.

## pi does not ask

Most agents ask for your permission before they run a command or change a
file. You then answer allow or deny, and you can only answer well if you can
read the command. Three questions for every command, from a human or an AI:

1. Which files does it **read**?
2. Which files does it **change or delete**?
3. Does it use the **network**: bring something in, or send something out?

**pi does not ask.** It runs commands and changes files straight away, with
all the rights of your account. So you answer the three questions
**afterwards**, and you need a way to undo. You already have one: git.

**The rule: commit before you let an agent change your code.** Afterwards:

- `git status` shows which files it changed or made,
- `git diff` shows exactly what it changed,
- `git restore FILE` brings back the committed version, if you do not like
  the change.

Work with pi only in a directory that is a git repository with everything
committed. And only on firehorse, which is a course computer with nothing
important on it.

## The API key

pi runs on firehorse, but the language model runs on a university server
behind openwebui. pi needs a way to say "I am allowed to use the models".
That is an **API key**: a long secret string, like a password for programs.
(API means *application programming interface*: a way for programs, not
people, to use a service.)

Make your key in the browser:

1. Log in to https://openwebui.uni-freiburg.de .
2. Click your name (bottom left), then **Settings**, then **Account**.
3. Under **API keys**, create a new secret key. Copy it. It starts with
   `sk-`.

Treat the key like a password:

- Anyone who has it can use the models in your name.
- Never put it into a notebook, a git commit, a chat, or a screenshot.
- If it has leaked, delete it in openwebui and make a new one.

## Install pi

pi has an official installer. You know this pattern from `uv` in week 6: a
program downloaded from the internet and run at once. We trust the source,
https://pi.dev , and we do it once, on purpose.

On firehorse:

```
firehorse$ curl -fsSL https://pi.dev/install.sh | sh
```

After a short animation, the installer asks questions. Answer like this:

- `Pi needs Node.js 22.19.0 or newer and npm. Install them now with standalone Node.js? [Y/n]`:
  press Enter (yes). pi is written in JavaScript, and Node.js is the program
  that runs JavaScript. It is installed into your home directory.
- If it asks `Add ... to your PATH ...? [Y/n]`: press Enter (yes).
- `Start pi now? [Y/n]`: type `n` and Enter. pi does not know your key yet.

The installation takes a minute or two. Then check:

```
firehorse$ pi --version
```

If this says `pi: command not found`, log out (`exit`) and log in again,
then try once more.

## Tell pi about the models

pi reads a file `~/.pi/agent/models.json`. It says where the models are and
which key to use. The installer has already made the directory
`~/.pi/agent`. To be safe, make sure it exists (`mkdir -p` does nothing if
it is already there), then make the file with `nano`:

```
firehorse$ mkdir -p ~/.pi/agent
firehorse$ nano ~/.pi/agent/models.json
```

Type or paste this, and replace `sk-01234567` with **your** key:

```json
{
  "providers": {
    "openwebui": {
      "baseUrl": "https://openwebui.uni-freiburg.de/api",
      "api": "openai-completions",
      "apiKey": "sk-01234567",
      "models": [
        { "id": "standard-coding-ufr" },
        { "id": "gemma-4-31b-llmlb" },
        { "id": "glm-5.3-flash-llmlb" },
        { "id": "qwen-3.8-27b-llmlb" },
        { "id": "openai/gpt-6.1-sol-llmlb" }
      ]
    }
  }
}
```

Save and exit (Ctrl-O, Enter, Ctrl-X). This is **JSON**, a common text
format for settings. It looks like Python dictionaries and lists, and it is
strict: every `"` and `,` matters, and there must be no `,` after the last
item of a list.

The first model in the list, `standard-coding-ufr`, is the one pi uses
when it starts.

Your key is in this file now. Make the file readable only for you:

```
firehorse$ chmod 600 ~/.pi/agent/models.json
firehorse$ ls -l ~/.pi/agent/models.json
-rw------- 1 yourname yourname 420 Dec  1 14:30 /home/yourname/.pi/agent/models.json
```

`-rw-------` means: you may read and write it, nobody else may do
anything.

## A first test

`pi -p` sends one request, prints the answer, and stops:

```
firehorse$ pi -p "Say hello in one short sentence."
Hello! How can I help you today?
```

Your answer will be different (remember lesson 1: the answers are random).
The university models can be slow. Wait up to a minute.

`pi --list-models` lists the models from your `models.json`.

## Working with pi

Make a directory for a small project, make it a git repository, and commit
a first file:

```
firehorse$ mkdir -p ~/week-07/hello
firehorse$ cd ~/week-07/hello
firehorse$ git init
firehorse$ echo "A greeting script" > README.md
firehorse$ git add README.md
firehorse$ git commit -m "Add README"
```

(`echo "..." > README.md` writes the text into the file, as in week 5.)
Now start pi in this directory:

```
firehorse$ pi
```

pi fills the terminal. At the bottom is a box where you type, and below it
a line with the directory and the model. Type a task and press Enter:

```
Write a Python script hello.py that prints "Hello, penguin!". Then run it.
```

Watch what pi does. It shows each tool it uses: here probably `write` for
the file and `bash` for `python3 hello.py`. For each `bash` command, ask the
three questions.

A few things you can type in pi:

| Type | What it does |
| --- | --- |
| a task, then Enter | give pi a task |
| `/model` | choose another model |
| `/new` | start a new conversation |
| `/quit` | leave pi |

When pi is done, type `/quit`. Then check what it did:

```
firehorse$ git status
firehorse$ cat hello.py
firehorse$ python3 hello.py
```

If you are happy with it, commit it:

```
firehorse$ git add hello.py
firehorse$ git commit -m "Add hello.py, written by pi"
```

Writing "written by pi" in the message is a good habit: it records how the
code was made.

## When something goes wrong

- `pi: command not found`: log out and in again. If that does not help,
  run the installer again.
- `401` and "token is invalid": the API key in `models.json` is wrong.
  Check that you copied all of it, and that it is between the `"`.
- An error that mentions JSON: there is a typo in `models.json`. Look for a
  missing `"` or `,`, or a `,` too many before `]` or `}`.
- pi works on the wrong files: check where you started it (`pwd`). pi
  works in the directory where you started it.
- pi is slow or answers nonsense: the university models are smaller than
  the best commercial ones. Make the task smaller and clearer, start again
  with `/new`, or try another model with `/model`.

## Summary

| When | Command | What it does |
| --- | --- | --- |
| once | `curl -fsSL https://pi.dev/install.sh \| sh` | install pi |
| once | `mkdir -p ~/.pi/agent`, then `nano ~/.pi/agent/models.json` | tell pi the models and your API key |
| once | `chmod 600 ~/.pi/agent/models.json` | only you may read the file with your key |
| | `pi -p "..."` | one request, one answer |
| | `pi` | start pi in the current directory |
| in pi | `/model`, `/new`, `/quit` | change model, new conversation, leave |
| before | `git commit` | a version to come back to |
| after | `git status`, `git diff` | what did the agent change? |
| if needed | `git restore FILE` | undo the agent's change to `FILE` |

- An agent runs commands and changes files itself. pi does not ask first.
- Commit before, look with `git status` and `git diff` after.
- Keep the API key secret.

## Going further (optional)

- Ask pi to explain a script you wrote yourself in week 5 or 6. Copy the
  script into a new repository first.
- If a directory contains a file `AGENTS.md`, pi reads it before every
  task. Write rules into it, for example "Use only the Python standard
  library." Does pi follow them?
- pi's documentation: https://pi.dev . It has a page on running pi safely.

## Check yourself

Answer these without a terminal first.

1. You let pi work in a directory. Afterwards, `git status` shows
   `modified: gc.py` and, under "Untracked files", `test_gc.py`. What did pi
   do?
2. Anna asks pi to "tidy up the directory". pi runs `rm *.txt`. Her data
   files were `.txt` files that she never committed. Can `git restore` bring
   them back? Why, or why not?
3. Ben pastes his `models.json`, with his key, into the course forum to ask
   for help. What should he do now?

Answers are at the end of this notebook.

## Answers to check yourself

1. It changed the existing file `gc.py` and made a new file `test_gc.py`.
   `git diff gc.py` shows the changes; `cat test_gc.py` shows the new file.
2. No. git can only bring back what was committed. This is why you commit
   *before* you let an agent work.
3. Delete that key in openwebui at once, make a new one, and put the new
   one into his `models.json`. Next time he replaces the key with
   `sk-...` before he shares the file.